bouncing_ball.ipynb
---------------------------
Overlay of u(t), v(t), and impulses from three integration schemes (NS-Newmark, CD-Lagrange, Moreau-Jean) on a single set of plots with identical parameters. Includes analytical u(t), v(t).

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, Tuple
import numpy as np
import plotly.graph_objects as go
import plotly.colors as pc
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
@dataclass
class Params:
    m: float = 1.0  # mass [kg]
    e: float = 0.8  # restitution coefficient in [0,1]
    g_const: float = 9.81  # gravity acceleration [m/s^2]
    k: float = 1e4  # penalty stiffness [N/m]
    c: float = 0.0  # contact damping [N·s/m]
    dt: float = 1e-2  # time step [s]
    t_end: float = 5.0  # end time [s]


# ─── Analytic bouncing ball sampler (u,v over time grid) ───
def _first_impact(u0: float, v0: float, g: float) -> Tuple[float, float]:
    D = v0 * v0 + 2.0 * g * u0 # noqa: N806
    if D < 0.0:
        raise ValueError("No impact: negative discriminant")
    t1 = (v0 + np.sqrt(D)) / g
    if t1 <= 0.0:
        raise ValueError("No future impact")
    v1m = v0 - g * t1
    return t1, v1m


def impact_schedule(
    u0: float,
    v0: float,
    g: float,
    e: float,
    t_end: float,
    tol: float = 1e-12,
    max_bounces: int = 10000,
) -> Tuple[np.ndarray, np.ndarray, bool]:
    T = [0.0] # noqa: N806
    u_up = []
    extinct = False
    try:
        t1, v1m = _first_impact(u0, v0, g)
    except ValueError:
        return np.array(T), np.array(u_up), extinct
    T.append(t1)
    u = e * abs(v1m)
    t = t1
    for _ in range(max_bounces):
        dt = 2.0 * u / g
        if dt < tol:
            extinct = True
            break
        t_next = t + dt
        u_up.append(u)
        T.append(t_next)
        if t_next >= t_end - tol:
            break
        u *= e
        t = t_next
    return np.asarray(T), np.asarray(u_up), extinct


def analytic_uv(
    t: np.ndarray, u0: float, v0: float, p: Params
) -> Dict[str, np.ndarray]:
    t = np.asarray(t, dtype=float)
    if t.size == 0:
        z = np.asarray([])
        return {"u_a": z, "v_a": z}
    g = p.g_const
    T, u_up, extinct = impact_schedule(u0, v0, g, p.e, float(t[-1])) # noqa: N806
    if len(T) == 1:
        u = u0 + v0 * t - 0.5 * g * t**2
        v = v0 - g * t
        return {"u_a": u, "v_a": v}
    t_last = T[-1]
    after_last = t >= t_last
    u = np.empty_like(t)
    v = np.empty_like(t)
    if extinct and np.any(after_last):
        u[after_last] = 0.0
        v[after_last] = 0.0
    active = ~after_last if extinct else np.ones_like(t, dtype=bool)
    if np.any(active):
        k = np.searchsorted(T, t[active], side="right") - 1
        k = np.clip(k, 0, len(T) - 2)
        s = t[active] - T[k]
        u_seg = np.empty_like(s)
        v_seg = np.empty_like(s)
        m0 = k == 0
        if np.any(m0):
            u_seg[m0] = u0 + v0 * s[m0] - 0.5 * g * s[m0] ** 2
            v_seg[m0] = v0 - g * s[m0]
        m1 = ~m0
        if np.any(m1):
            up = u_up[k[m1] - 1]
            u_seg[m1] = up * s[m1] - 0.5 * g * s[m1] ** 2
            v_seg[m1] = up - g * s[m1]
        u[active] = u_seg
        v[active] = v_seg
    return {"u_a": u, "v_a": v}


# ─── Numerical schemes (return t, u, v, impulse arrays) ───
def simulate_ns_newmark(u0: float, v0: float, p: Params) -> Dict[str, np.ndarray]:
    u, v, a = float(u0), float(v0), -p.g_const
    dt = p.dt
    t = 0.0
    T, U, V, IMP = [t], [u], [v], [0.0] # noqa: N806
    n_steps = int(np.ceil(p.t_end / dt))
    for _ in range(n_steps):
        u_tilde = u + dt * v + 0.5 * dt * dt * a
        a_next = -p.g_const
        v_tilde = v + 0.5 * dt * (a + a_next)

        if (u_tilde <= 0.0) and ((1 + p.e) * v - dt * p.g_const < 0.0):
            v_next = -p.e * v
            w_next = v_next - v_tilde  # velocity jump due to impulse
            imp_next = p.m * w_next
            u_next = u_tilde + 0.5 * dt * w_next  # Newmark-compatible correction
        else:
            v_next = v_tilde
            imp_next = 0.0
            u_next = u_tilde
        u, v, a = u_next, v_next, a_next
        t += dt
        T.append(t)
        U.append(u)
        V.append(v)
        IMP.append(imp_next)
    return {
        "t": np.asarray(T),
        "u": np.asarray(U),
        "v": np.asarray(V),
        "imp": np.asarray(IMP),
    }


def simulate_cd_lagrange(u0: float, v0: float, p: Params) -> Dict[str, np.ndarray]:
    dt = p.dt
    vhalf = v0 + 0.5 * dt * (-p.g_const)
    u = float(u0)
    t = 0.0
    T, U, Vc, IMP = [t], [u], [v0], [0.0] # Vc: centered integer-time velocity # noqa: N806
    n_steps = int(np.ceil(p.t_end / dt))
    for _ in range(n_steps):
        u_next = u + dt * vhalf
        vhalf_free_next = vhalf - dt * p.g_const
        if (u_next <= 0.0) and ((vhalf_free_next + p.e * vhalf) < 0.0):
            vhalf_next = -p.e * vhalf
            imp_next = p.m * (vhalf_next - vhalf_free_next)
        else:
            vhalf_next = vhalf_free_next
            imp_next = 0.0
        # centered velocity at t_{n+1}
        v_center_next = 0.5 * (vhalf + vhalf_next)
        u, vhalf = u_next, vhalf_next
        t += dt
        T.append(t)
        U.append(u)
        Vc.append(v_center_next)
        IMP.append(imp_next)
    return {
        "t": np.asarray(T),
        "u": np.asarray(U),
        "v": np.asarray(Vc),
        "imp": np.asarray(IMP),
    }


def simulate_moreau_jean(u0: float, v0: float, p: Params) -> Dict[str, np.ndarray]:
    dt = p.dt
    u, v = float(u0), float(v0)
    t = 0.0
    T, U, V, IMP = [t], [u], [v], [0.0] # noqa: N806
    n_steps = int(np.ceil(p.t_end / dt))
    for _ in range(n_steps):
        #g_half = u + 0.5 * dt * v
        g_full = u + dt * v
        v_free = v - dt * p.g_const
        #if g_half <= 0.0 and (v_free + p.e * v) < 0.0:
        if g_full <= 0.0 and (v_free + p.e * v) < 0.0:
            v_next = -p.e * v
            imp_next = p.m * (v_next - v_free)
        else:
            v_next = v_free
            imp_next = 0.0
        u_next = u + 0.5 * dt * (v + v_next)
        u, v = u_next, v_next
        t += dt
        T.append(t)
        U.append(u)
        V.append(v)
        IMP.append(imp_next)
    return {
        "t": np.asarray(T),
        "u": np.asarray(U),
        "v": np.asarray(V),
        "imp": np.asarray(IMP),
    }


def simulate_penalty(u0: float, v0: float, p: Params) -> Dict[str, np.ndarray]:
    """
    Central-difference with penalty contact.
    Returns per-step contact impulse as Fc * dt.
    """
    dt = p.dt
    u = float(u0)
    v = float(v0)
    a = -p.g_const
    t = 0.0
    T, U, V, IMP = [t], [u], [v], [0.0] # noqa: N806
    n_steps = int(np.ceil(p.t_end / dt))
    for _ in range(n_steps):
        u_pred = u + dt * v + 0.5 * dt**2 * a
        v_pred = v + dt * a
        if u_pred <= 0.0:
            fc = -p.k * u_pred - p.c * v_pred
        else:
            fc = 0.0
        a_new = -p.g_const + fc / p.m
        v_new = v + 0.5 * dt * (a + a_new)
        u_new = u_pred
        t += dt
        T.append(t)
        U.append(u_new)
        V.append(v_new)
        IMP.append(fc * dt)
        u, v, a = u_new, v_new, a_new
    return {
        "t": np.asarray(T),
        "u": np.asarray(U),
        "v": np.asarray(V),
        "imp": np.asarray(IMP),
    }

In [ ]:
# ─── Plot layout helper ───

def get_layout() -> go.Layout:

    layout = go.Layout(
        xaxis=dict(
            showgrid=True,
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            zeroline=False,
            ticks="inside",
            exponentformat="power",
            showexponent="last",
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            showgrid=True,
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            zeroline=False,
            ticks="inside",
            exponentformat="power",
            tickfont=dict(size=12),
        ),
        font=dict(family="Latin-Modern", size=12, color="Black"),
        legend=dict(
            x=0.97,  # position from the left (0 to 1)
            y=1.2,  # position from the bottom (0 to 1)
            bgcolor="rgba(255, 255, 255, 0.8)",  # white with 80% opacity
            bordercolor="black",
            borderwidth=1,
            orientation="h",
            xanchor="right",
            yanchor="top",
        ),
        width=600,
        height=500,
        showlegend=True,
        template="plotly_white",
    )

    return layout

In [ ]:
# ─── Simulation and plotting parameters ───

# Common parameters and initial conditions
p = Params(m=1, e=0.8,  g_const=9.81, dt=1e-2, t_end=5.0)
u0, v0 = 1.0, 0.0

res = {
    "Moreau-Jean": simulate_moreau_jean(u0, v0, p),
    "CD-Lagrange": simulate_cd_lagrange(u0, v0, p),
    "NS-Newmark": simulate_ns_newmark(u0, v0, p),
}

names = list(res.keys())
vals = np.linspace(0.0, 0.6, len(names))

colors = {}
colors["Moreau-Jean"] = "lightblue"
colors["CD-Lagrange"] = "#2072b2"
colors["NS-Newmark"] = "red"
dashs = {
    "Moreau-Jean": "solid",
    "CD-Lagrange": "solid",
    "NS-Newmark": "dash",
}

In [ ]:
# ─── Plotting ───

# Common time grid
t_end_common = min(float(r["t"][-1]) for r in res.values())
t_common = np.linspace(0.0, t_end_common, len(res["NS-Newmark"]["t"]))

# Analytic reference on common grid
ana = analytic_uv(t_common, u0, v0, p)

# Displacement plot with analytic
fig_u = go.Figure()
fig_u.add_trace(
    go.Scatter(
        x=t_common,
        y=ana["u_a"],
        mode="lines",
        name="Analytic",
        line=dict(color="black", dash="solid", width=2),
    )
)
for name, r in res.items():
    fig_u.add_trace(
        go.Scatter(
            x=t_common,
            y=r["u"],
            mode="lines",
            name=name,
            line=dict(color=colors[name], dash=dashs[name], width=2),
        )
    )
fig_u.update_layout(
    title="Bouncing ball: u(t) overlay",
    xaxis_title=r"$t \mathrm{ (s)}$",
    yaxis_title=r"$u \mathrm{ (m)}$",
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1.0),
)
fig_u.update_layout(get_layout())

fig_u.show()

# Displacement difference w.r.t. analytic
fig_du = go.Figure()
for name, r in res.items():
    fig_du.add_trace(
        go.Scatter(
            x=t_common,
            y=(r["u"] - ana["u_a"]),
            mode="lines",
            name=name,
            line=dict(color=colors[name], dash=dashs[name]),
        )
    )
fig_du.update_layout(get_layout())
fig_du.update_layout(
    title="Bouncing ball: displacement error",
    xaxis_title=r"$t \mathrm{ (s)}$",
    yaxis_title=r"$u - u_{analytic} \mathrm{ (m)}$",
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1.0),
)
fig_du.show()

# Velocity plot with analytic
fig_v = go.Figure()
fig_v.add_trace(
    go.Scatter(
        x=t_common,
        y=ana["v_a"],
        mode="lines",
        name="Analytic",
        line=dict(color="black", dash="solid", width=2),
    )
)
for name, r in res.items():
    fig_v.add_trace(
        go.Scatter(
            x=t_common,
            y=r["v"],
            mode="lines",
            name=name,
            line=dict(color=colors[name], dash=dashs[name], width=2),
        )
    )
fig_v.update_layout(get_layout())
fig_v.update_layout(
    title="Bouncing ball: v(t) overlay",
    xaxis_title=r"$t \mathrm{ (s)}$",
    yaxis_title=r"$v \mathrm{ (m/s)}$",
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1.0),
)
fig_v.show()

# Velocity difference w.r.t. analytic
fig_dv = go.Figure()
for name, r in res.items():
    fig_dv.add_trace(
        go.Scatter(
            x=t_common,
            y=r["v"] - ana["v_a"],
            mode="lines",
            name=name,
            line=dict(color=colors[name], dash=dashs[name], width=2),
        )
    )
fig_dv.update_layout(get_layout())
fig_dv.update_layout(
    title="Bouncing ball: velocity error",
    xaxis_title=r"$t \mathrm{ (s)}$",
    yaxis_title=r"$v - v_{analytic} \mathrm{ (m/s)}$",
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1.0),
)
fig_dv.show()

# Impulse plot
fig_imp = go.Figure()
for name, r in res.items():
    fig_imp.add_trace(
        go.Scatter(
            x=t_common,
            y=r["imp"],
            mode="lines",
            name=name,
            line=dict(color=colors[name], dash=dashs[name], width=2),
        )
    )
fig_imp.update_layout(get_layout())
fig_imp.update_layout(
    title="Contact impulse (per step)",
    xaxis_title=r"$t \mathrm{ (s)}$",
    yaxis_title=r"$\mathrm{impulse}$",
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1.0),
)
fig_imp.show()

In [ ]:
# ─── Convergence study: position error vs timestep (log-log) ───

# e(dt) = sum_i |u_i - u_a(t_i)| / sum_i |u_a(t_i)|
p = Params(m=1.0, e=1.0, g_const=9.81, dt=0.7e-2, t_end=5.0)
u0, v0 = 1.0, 0.0

write_output = False
output_name = f"bouncing_ball_nsn_{p.e:.2f}"

# Timesteps to test
dt_vals = np.logspace(-6, -1, 50)
max_steps = 1e6

def horizon_for(dt: float) -> float:
    return p.t_end


methods = {
    "Moreau-Jean": simulate_moreau_jean,
    "CD-Lagrange": simulate_cd_lagrange,
    "NS-Newmark": simulate_ns_newmark,
}

err = {name: [] for name in methods}

for dt in dt_vals:
    print(f"Testing dt={dt:.2e}"    )
    p_dt = Params(
        m=p.m, e=p.e, g_const=p.g_const, dt=float(dt), t_end=float(horizon_for(dt))
    )
    for name, sim in methods.items():
        r = sim(u0, v0, p_dt)
        t = r["t"]
        u = r["u"]
        a = analytic_uv(t, u0, v0, p_dt)["u_a"]
        denom = np.sum(np.abs(a))
        num = np.sum(np.abs(u - a))
        e = (num / denom) if denom > 0 else np.inf
        err[name].append(e)

In [ ]:
# ─── Convergence study: Plotting ───

# Plot on log-log axes
fig_conv = go.Figure()
for name, e_list in err.items():
    fig_conv.add_trace(
        go.Scatter(
            x=dt_vals,
            y=np.asarray(e_list),
            mode="lines",
            name=name,
            line=dict(color=colors[name], dash=dashs[name], width=2),
        )
    )

# Add O(dt) reference line scaled to mid-range
mid = len(dt_vals) // 2
ref_level = (
    2 * np.median([err[k][mid] for k in methods])
    if all(len(err[k]) > mid for k in methods)
    else 1.0
)
C = ref_level / dt_vals[mid]
fig_conv.add_trace(
    go.Scatter(
        x=dt_vals[-10:-5],
        y=C * dt_vals[-10:-5],
        mode="lines",
        name=r"$\mathcal{O}(\Delta t)$",
        line=dict(color="black"),
    )
)

fig_conv.update_layout(get_layout())
fig_conv.update_layout(
    title="Convergence of position error vs timestep",
    xaxis_title=r"$\Delta t \mathrm{ (s)}$",
    yaxis_title=r"$\eta$",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0),
)
fig_conv.update_xaxes(type="log")
fig_conv.update_yaxes(type="log", range=[-6.5, 0])
fig_conv.show()